# Milestone 2 — Data Model
## Delhi Air Quality Dataset
**Team 5** | Database: SQLite (Relational / Star Schema)

## 1. Why SQLite (RDBMS)?
### Evaluation of storage options
| Criterion | RDBMS (SQLite) | NoSQL (MongoDB) | Wide-Column (Cassandra) |
|---|---|---|---|
| Schema variability | Fixed ✓ | Variable | Fixed |
| Relational structure | Strong ✓ | Weak | None |
| Query pattern | SQL analytics ✓ | Document lookup | Write-heavy IoT |
| Sensor count | 15 ✓ | Any | 1000s preferred |
| Infra complexity | None ✓ | Server required | Cluster required |

**Conclusion:** Our data is structured, narrow, relational, and read-heavy — RDBMS wins.
- **Not NoSQL**: Schema is fixed, not document-variable
- **Not Wide-Column**: Only 15 sensors (Cassandra shines at thousands), analytics > writes
- **Not flat/wide table**: Station name/city/state duplicated 8.8M times → normalise

## 2. Star Schema Design


In [ ]:
import sqlite3, pandas as pd, glob, re
from pathlib import Path

BASE_DIR = Path(".").resolve()
DB_PATH  = BASE_DIR / "data" / "database" / "air_quality.db"
conn = sqlite3.connect(DB_PATH)
print("Connected to:", DB_PATH)

## 3. Row Counts per Table

In [ ]:
tables = ["dim_station", "dim_pollutant", "dim_time", "fact_measurements"]
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<25}: {n:>10,}")

## 4. Inspect Dimension Tables

In [ ]:
print("── dim_station ──")
pd.read_sql("SELECT * FROM dim_station", conn)

In [ ]:
print("── dim_pollutant ──")
pd.read_sql("SELECT * FROM dim_pollutant", conn)

In [ ]:
print("── dim_time sample ──")
pd.read_sql("SELECT * FROM dim_time LIMIT 10", conn)

## 5. Sample Analytical Queries

In [ ]:
# Q1: Average PM2.5 per station
q = """
SELECT s.station_name,
       ROUND(AVG(f.value), 2) AS avg_pm25,
       ROUND(MIN(f.value), 2) AS min_pm25,
       ROUND(MAX(f.value), 2) AS max_pm25
FROM   fact_measurements f
JOIN   dim_station   s ON s.station_id   = f.station_id
JOIN   dim_pollutant p ON p.pollutant_id = f.pollutant_id
WHERE  p.pollutant_name = 'pm25'
GROUP  BY s.station_name
ORDER  BY avg_pm25 DESC
"""
pd.read_sql(q, conn)

In [ ]:
# Q2: Hourly avg temperature per station
q = """
SELECT t.hour, s.station_name, ROUND(AVG(f.at_c),2) AS avg_temp_c
FROM   fact_measurements f
JOIN   dim_station s ON s.station_id = f.station_id
JOIN   dim_time    t ON t.time_id    = f.time_id
WHERE  f.at_c IS NOT NULL
GROUP  BY t.hour, s.station_name
ORDER  BY t.hour, s.station_name
LIMIT  30
"""
pd.read_sql(q, conn)

In [ ]:
# Q3: Pollutant measurement counts (data completeness)
q = """
SELECT p.pollutant_name,
       COUNT(*) AS total_readings,
       SUM(CASE WHEN f.value IS NULL THEN 1 ELSE 0 END) AS null_count
FROM   fact_measurements f
JOIN   dim_pollutant p ON p.pollutant_id = f.pollutant_id
GROUP  BY p.pollutant_name ORDER BY total_readings DESC
"""
pd.read_sql(q, conn)

In [ ]:
conn.close()
print("Done")